## PATCH applied (for Reviewer 1, Comment 8 -- natural class distribution)

Three additive changes only. **Nothing that produces the existing balanced CSV was modified or removed.**

1. `build_dataset()` now also returns the full, un-undersampled arrays (`X_full, y_full, meta_full`) that already existed in memory before the random undersampling step -- they were simply discarded before.
2. A new function `save_unbalanced_csv()` writes those arrays to a second CSV, with the natural class ratio preserved (no undersampling).
3. `main()` calls the new function after the existing `save_csv()` call, saving `unbalanced_dataset_22ch_924feat_with_subjects.csv` next to the existing output file.

Just run all cells as normal. The original balanced CSV will be produced exactly as before, and the new unbalanced CSV will appear alongside it.

In [22]:
#!pip install mne
#!pip install PyWavelets

In [23]:

import re, math, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm
import mne

from scipy.signal import welch, hilbert
from scipy.stats  import skew, kurtosis
import pywt

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")


# ======================== CONFIG ================================
CFG = dict(
    data_dir   = r"F:\submitted\epilipsy\code\data_Seizure",
    # Renamed on purpose -- this run adds subject_id/event_id and uses
    # 22 channels (924 features), matching the manuscript. Do NOT overwrite
    # the old balanced_dataset8.csv (23 channels, 966 features, no
    # subject_id) -- that file is a different, incompatible feature set.
    output_csv = r"F:\submitted\epilipsy\code\results\balanced_dataset_22ch_924feat_with_subjects.csv",

    sfreq      = 256,
    n_channels = 22,

    epoch_len           = 2,     # seconds → 512 samples
    ictal_overlap       = 0.90,  # stride = 51 samples (~0.2 s)
    interictal_overlap  = 0.50,  # stride = 256 samples (1.0 s)

    pre_ictal_ext  = 10,         # seconds before onset  → ictal mask
    post_ictal_ext =  5,         # seconds after offset  → ictal mask

    target_each = 100_000,       # cap per class; auto-reduced to real ictal count

    wavelet       = "db4",
    wavelet_level = 4,
    perm_order    = 3,
    welch_nperseg = 128,

    # ── LIF neuron parameters ────────────────────────────────────
    # Tuned for µV-scale EEG signals
    lif_tau      = 0.020,   # membrane time constant (s)  — 20 ms
    lif_r        = 1.0,     # membrane resistance (normalised)
    lif_v_th     = 1.0,     # firing threshold (applied to z-scored signal)
    lif_v_rest   = 0.0,     # resting / reset potential

    # Burst detection: two consecutive spikes are a "burst"
    # if their ISI < burst_thr seconds
    lif_burst_thr = 0.050,  # 50 ms ISI threshold for burst classification

    # Spike-count entropy bin width
    lif_bin_ms   = 50,      # 50 ms bins for spike-count histogram
)


# ======================== FEATURE NAMES (42) ====================
# 34 original + 8 SNN features per channel
FEAT_NAMES = [
    # ── Time-domain (8) ─────────────────────────────────────────
    "skewness",           # 01
    "kurtosis",           # 02
    "zcr",                # 03
    "ptp",                # 04  computed on RAW signal
    "curve_len",          # 05
    "hjorth_mob",         # 06
    "hjorth_comp",        # 07
    "iqr",                # 08
    # ── Frequency (11) ──────────────────────────────────────────
    "delta_pow",          # 09
    "theta_pow",          # 10
    "alpha_pow",          # 11
    "beta_pow",           # 12
    "gamma_pow",          # 13
    "mean_freq",          # 14
    "dom_freq",           # 15
    "spec_entropy",       # 16
    "alpha_beta_ratio",   # 17
    "spec_flatness",      # 18
    "spec_edge_95",       # 19
    # ── Wavelet db4 level-4 (8) ─────────────────────────────────
    "wav_e1",             # 20
    "wav_e2",             # 21
    "wav_e3",             # 22
    "wav_e4",             # 23
    "wav_entropy",        # 24
    "wav_total_e",        # 25
    "wav_hf_ratio",       # 26
    "wav_std",            # 27
    # ── Nonlinear (7) ───────────────────────────────────────────
    "perm_entropy",       # 28
    "petrosian_fd",       # 29
    "nleo_mean",          # 30
    "nleo_std",           # 31
    "hilbert_amp",        # 32
    "hilbert_phase_std",  # 33
    "crest_factor",       # 34  computed on RAW signal
    # ── SNN / LIF spike-train features (8) ──────────────────────
    "snn_spike_rate",     # 35  spikes / second
    "snn_mean_isi",       # 36  mean inter-spike interval (s)
    "snn_std_isi",        # 37  std of ISI
    "snn_cv_isi",         # 38  coefficient of variation of ISI
    "snn_burst_index",    # 39  fraction of spikes in bursts
    "snn_mean_burst_len", # 40  mean burst length (# spikes)
    "snn_silent_frac",    # 41  fraction of time below threshold
    "snn_spike_entropy",  # 42  entropy of spike-count histogram
]
assert len(FEAT_NAMES) == 42, f"Expected 42, got {len(FEAT_NAMES)}"
N_FEAT = len(FEAT_NAMES)


In [ ]:

# ======================== LOADER ================================
class CHBLoader:
    def __init__(self, root):
        self.root = Path(root)

    def _parse_summary(self, subj_dir, subj):
        f = subj_dir / f"{subj}-summary.txt"
        seizures, cur, start = {}, None, None
        if not f.exists():
            return seizures
        for line in open(f, encoding="utf-8", errors="ignore"):
            line = line.strip()
            if "File Name:" in line:
                cur = line.split(":", 1)[1].strip()
                seizures.setdefault(cur, [])
            if re.search(r"Seizure.*Start Time", line, re.I) and cur:
                nums = re.findall(r"[\d.]+", line.split(":")[-1])
                if nums: start = float(nums[0])
            if re.search(r"Seizure.*End Time", line, re.I) \
                    and cur and start is not None:
                nums = re.findall(r"[\d.]+", line.split(":")[-1])
                if nums:
                    seizures[cur].append((start, float(nums[0])))
                    start = None
        return seizures

    def load(self):
        records = []
        for subj in sorted(self.root.iterdir()):
            if not subj.is_dir():
                continue
            summary = self._parse_summary(subj, subj.name)
            for edf in sorted(subj.glob("*.edf")):
                try:
                    raw = mne.io.read_raw_edf(
                        str(edf), preload=True, verbose=False)
                    if abs(raw.info["sfreq"] - CFG["sfreq"]) > 1:
                        raw.resample(CFG["sfreq"], verbose=False)
                    records.append({
                        "raw":           raw,
                        "seizure_times": summary.get(edf.name, []),
                        "filename":      edf.name,
                    })
                except Exception as e:
                    print(f"  [skip] {edf.name}: {e}")
        print(f"  Loaded {len(records)} EDF files.")
        return records


# ======================== HELPERS ================================
def _safe(x):
    v = float(x)
    return 0.0 if (v != v or abs(v) == float("inf")) else v


def _perm_entropy(sig, order=3):
    """Normalised permutation entropy — fully vectorised."""
    N = len(sig)
    if N < order:
        return 0.0
    shape   = (N - order + 1, order)
    strides = (sig.strides[0], sig.strides[0])
    X    = np.lib.stride_tricks.as_strided(sig, shape=shape, strides=strides)
    idx  = np.argsort(X, axis=1, kind="quicksort")
    _, c = np.unique(idx @ (order ** np.arange(order, dtype=np.int64)),
                     return_counts=True)
    p    = c / c.sum()
    return _safe(-np.sum(p * np.log(p + 1e-12)) / math.log(order))


def _petrosian_fd(sig):
    """Petrosian fractal dimension — O(N), fully vectorised."""
    N  = len(sig)
    Nd = int(np.count_nonzero(np.diff(np.sign(np.diff(sig)))))
    if Nd == 0:
        return 0.0
    return _safe(math.log10(N) /
                 (math.log10(N) + math.log10(N / (N + 0.4 * Nd))))


# ======================== SNN / LIF FEATURES ====================
def lif_spike_features(sig_raw, fs):
    """
    Simulate a Leaky Integrate-and-Fire (LIF) neuron driven by the
    EEG channel signal and extract 8 spike-train statistics.

    The LIF dynamics in discrete time (dt = 1/fs):
        V[t] = V[t-1] * decay + I[t]
        if V[t] >= V_th:  spike → V[t] = V_rest

    where:
        decay = exp(-dt / tau)
        I[t]  = z-scored signal sample (input current)

    Parameters
    ----------
    sig_raw : np.ndarray (L,) — raw (non-normalised) EEG channel
    fs      : int — sampling frequency

    Returns
    -------
    list of 8 floats
    """
    eps = 1e-9
    N   = len(sig_raw)
    dt  = 1.0 / fs

    # Input current = z-scored signal (keeps scale-invariance)
    mu  = sig_raw.mean()
    sd  = sig_raw.std() + eps
    I   = (sig_raw - mu) / sd

    # LIF parameters
    tau       = CFG["lif_tau"]
    v_th      = CFG["lif_v_th"]
    v_rest    = CFG["lif_v_rest"]
    decay     = math.exp(-dt / tau)
    burst_thr = CFG["lif_burst_thr"]
    bin_sz    = max(1, int(CFG["lif_bin_ms"] * 1e-3 * fs))  # samples per bin

    # ── Vectorised LIF simulation ────────────────────────────────
    # Pure-NumPy implementation; no explicit Python time loop.
    # Uses cumulative sum trick for the leaky integration between
    # threshold crossings — valid because decay is constant.
    #
    # Algorithm:
    #   1. Compute raw integrated potential ignoring resets
    #   2. Detect threshold crossings iteratively (unavoidable for
    #      reset logic) but keep the inner body trivial.
    #
    V       = np.empty(N, dtype=np.float64)
    spikes  = np.zeros(N, dtype=np.uint8)
    v       = v_rest

    # We use a small Numba-free loop — but it's only over N=512 steps
    # (2 s × 256 Hz), so it takes < 1 ms per channel in CPython.
    for t in range(N):
        v = v * decay + I[t]
        if v >= v_th:
            spikes[t] = 1
            v = v_rest
        V[t] = v

    # ── Spike-train statistics ───────────────────────────────────
    spike_times = np.where(spikes == 1)[0]     # sample indices of spikes
    n_spikes    = len(spike_times)
    duration    = N / fs                       # window duration in seconds

    # 35 — Spike rate (Hz)
    spike_rate = _safe(n_spikes / (duration + eps))

    if n_spikes < 2:
        # No meaningful ISI statistics — return safe defaults
        return [
            spike_rate,   # snn_spike_rate
            0.0,          # snn_mean_isi
            0.0,          # snn_std_isi
            0.0,          # snn_cv_isi
            0.0,          # snn_burst_index
            0.0,          # snn_mean_burst_len
            _safe(float(np.mean(V < v_th))),  # snn_silent_frac
            0.0,          # snn_spike_entropy
        ]

    # Inter-spike intervals (in seconds)
    isi    = np.diff(spike_times).astype(np.float64) / fs
    m_isi  = float(isi.mean())
    s_isi  = float(isi.std())
    cv_isi = _safe(s_isi / (m_isi + eps))

    # 39 — Burst index: fraction of spikes preceded by ISI < burst_thr
    burst_mask   = isi < burst_thr            # True for each ISI in a burst
    # spike[0] can't have a preceding ISI; spikes[1:] correspond to isi[:]
    in_burst     = np.concatenate([[False], burst_mask])
    burst_index  = _safe(float(in_burst.sum()) / n_spikes)

    # 40 — Mean burst length: run-length encoding of consecutive burst spikes
    burst_lengths = []
    run = 0
    for b in in_burst:
        if b:
            run += 1
        else:
            if run > 0:
                burst_lengths.append(run + 1)   # +1 for the initiating spike
                run = 0
    if run > 0:
        burst_lengths.append(run + 1)
    mean_burst_len = _safe(float(np.mean(burst_lengths)) if burst_lengths else 1.0)

    # 41 — Silent fraction: proportion of time steps below threshold
    silent_frac = _safe(float(np.mean(V < v_th)))

    # 42 — Spike-count entropy over 50ms bins
    n_bins      = max(1, N // bin_sz)
    bin_counts  = np.array([
        spikes[i*bin_sz : (i+1)*bin_sz].sum()
        for i in range(n_bins)
    ], dtype=np.float64)
    bc_sum = bin_counts.sum() + eps
    p_bins = bin_counts / bc_sum
    spike_ent = _safe(float(-np.sum(p_bins * np.log(p_bins + 1e-12))))

    return [
        spike_rate,                  # 35 snn_spike_rate
        _safe(m_isi),                # 36 snn_mean_isi
        _safe(s_isi),                # 37 snn_std_isi
        cv_isi,                      # 38 snn_cv_isi
        burst_index,                 # 39 snn_burst_index
        mean_burst_len,              # 40 snn_mean_burst_len
        silent_frac,                 # 41 snn_silent_frac
        spike_ent,                   # 42 snn_spike_entropy
    ]


# ======================== FULL FEATURE EXTRACTION (42) ==========
def extract_features(sig_raw, fs):
    """
    Extract all 42 features from one EEG channel segment.

    sig_raw : np.ndarray (L,) float64 — raw µV signal
    fs      : int — sampling frequency

    Returns list[float] of length 42.

    Design
    ──────
    • ptp and crest_factor computed on raw signal (amplitude info)
    • SNN features computed on raw signal via LIF neuron simulation
    • Remaining 33 features computed on z-scored signal
    • Welch, DWT, Hilbert each called once and results shared
    """
    eps = 1e-9
    N   = len(sig_raw)

    # ── A. Raw-signal features (before normalisation) ────────────
    raw_ptp   = _safe(float(sig_raw.max() - sig_raw.min()))
    raw_rms   = math.sqrt(float((sig_raw * sig_raw).mean())) + eps
    raw_crest = _safe(float(np.max(np.abs(sig_raw))) / raw_rms)

    # ── B. SNN features (LIF simulation on raw signal) ───────────
    snn_feats = lif_spike_features(sig_raw, fs)   # 8 features

    # ── C. Z-score normalise for remaining features ───────────────
    sig = (sig_raw - sig_raw.mean()) / (sig_raw.std() + eps)

    d1     = np.diff(sig)
    d2     = np.diff(d1)
    sq     = sig * sig
    abs_d1 = np.abs(d1)

    # Welch PSD — shared for all spectral features
    freqs, psd = welch(sig, fs=fs,
                       nperseg=min(CFG["welch_nperseg"], N),
                       window="hann")
    psd_sum = psd.sum() + eps
    psd_n   = psd / psd_sum

    # DWT db4 level-4 — shared for all wavelet features
    coeffs  = pywt.wavedec(sig, CFG["wavelet"], level=CFG["wavelet_level"])
    det_e   = np.array([float(np.dot(c, c)) for c in coeffs[1:]])
    det_sum = det_e.sum() + eps

    # Hilbert — shared for amplitude + phase features
    an      = hilbert(sig)
    amp_env = np.abs(an)
    ph_unw  = np.unwrap(np.angle(an))

    # NLEO
    nleo = sq[1:-1] - sig[:-2] * sig[2:]

    # ── TIME-DOMAIN (8) ──────────────────────────────────────────
    h_act = float(np.var(sig))
    h_mob = float(math.sqrt(float(np.var(d1)) / (h_act + eps)))
    h_dm  = float(math.sqrt(float(np.var(d2)) / (float(np.var(d1)) + eps)))

    t = [
        _safe(float(skew(sig))),                                        # 01 skewness
        _safe(float(kurtosis(sig))),                                    # 02 kurtosis
        _safe(float(np.count_nonzero((sig[:-1]*sig[1:])<0) / (N-1))),  # 03 zcr
        raw_ptp,                                                        # 04 ptp
        _safe(float(abs_d1.sum())),                                     # 05 curve_len
        _safe(h_mob),                                                   # 06 hjorth_mob
        _safe(h_dm / (h_mob + eps)),                                    # 07 hjorth_comp
        _safe(float(np.percentile(sig, 75) - np.percentile(sig, 25))), # 08 iqr
    ]

    # ── FREQUENCY (11) ───────────────────────────────────────────
    def _bp(lo, hi):
        m = (freqs >= lo) & (freqs <= hi)
        return _safe(float(psd[m].mean())) if m.any() else 0.0

    cum_psd = np.cumsum(psd_n)
    i95     = int(np.searchsorted(cum_psd, 0.95))
    log_gm  = math.exp(float(np.mean(np.log(psd + eps))))
    f_alpha = _bp(8, 13)
    f_beta  = _bp(13, 30)

    f = [
        _bp(0.5, 4),                                                    # 09 delta
        _bp(4,   8),                                                    # 10 theta
        f_alpha,                                                        # 11 alpha
        f_beta,                                                         # 12 beta
        _bp(30,  70),                                                   # 13 gamma
        _safe(float(np.dot(freqs, psd_n))),                             # 14 mean_freq
        _safe(float(freqs[int(np.argmax(psd))])),                       # 15 dom_freq
        _safe(float(-np.dot(psd_n, np.log(psd_n + eps)))),             # 16 spec_entropy
        _safe(f_alpha / (f_beta + eps)),                                # 17 alpha_beta_ratio
        _safe(log_gm / (float(psd.mean()) + eps)),                      # 18 spec_flatness
        _safe(float(freqs[min(i95, len(freqs)-1)])),                    # 19 spec_edge_95
    ]

    # ── WAVELET (8) ──────────────────────────────────────────────
    p_e = det_e / det_sum
    w = [
        _safe(float(det_e[0])),                                         # 20 wav_e1
        _safe(float(det_e[1])),                                         # 21 wav_e2
        _safe(float(det_e[2])),                                         # 22 wav_e3
        _safe(float(det_e[3])),                                         # 23 wav_e4
        _safe(float(-np.dot(p_e, np.log(p_e + eps)))),                 # 24 wav_entropy
        _safe(float(det_sum)),                                          # 25 wav_total_e
        _safe(float((det_e[-1] + det_e[-2]) / det_sum)),                # 26 wav_hf_ratio
        _safe(float(np.mean([c.std() for c in coeffs[1:]]))),          # 27 wav_std
    ]

    # ── NONLINEAR (7) ────────────────────────────────────────────
    nl = [
        _perm_entropy(sig, CFG["perm_order"]),                          # 28 perm_entropy
        _petrosian_fd(sig),                                             # 29 petrosian_fd
        _safe(float(nleo.mean())),                                      # 30 nleo_mean
        _safe(float(nleo.std())),                                       # 31 nleo_std
        _safe(float(amp_env.mean())),                                   # 32 hilbert_amp
        _safe(float(ph_unw.std())),                                     # 33 hilbert_phase_std
        raw_crest,                                                      # 34 crest_factor
    ]

    row = t + f + w + nl + snn_feats   # 8+11+8+7+8 = 42
    assert len(row) == N_FEAT, f"Feature count: {len(row)} != {N_FEAT}"
    return row


# ======================== DATASET BUILDER =======================
def build_dataset(records):
    fs     = CFG["sfreq"]
    win    = int(CFG["epoch_len"] * fs)
    ic_str = max(1, int(win * (1 - CFG["ictal_overlap"])))
    ii_str = max(1, int(win * (1 - CFG["interictal_overlap"])))
    TARGET = CFG["target_each"]

    n_ch = min(
        min(r["raw"].get_data().shape[0] for r in records),
        CFG["n_channels"],
    )

    print(f"\n  Settings:")
    print(f"    Channels          : {n_ch}")
    print(f"    Window            : {win} samp ({CFG['epoch_len']} s)")
    print(f"    Ictal stride      : {ic_str} samp "
          f"({ic_str/fs*1000:.0f} ms — {CFG['ictal_overlap']*100:.0f}% overlap)")
    print(f"    Inter-ictal stride: {ii_str} samp "
          f"({ii_str/fs*1000:.0f} ms — {CFG['interictal_overlap']*100:.0f}% overlap)")
    print(f"    Mask extension    : -{CFG['pre_ictal_ext']}s / "
          f"+{CFG['post_ictal_ext']}s")
    print(f"    LIF tau={CFG['lif_tau']*1000:.0f}ms  "
          f"V_th={CFG['lif_v_th']}  "
          f"burst_thr={CFG['lif_burst_thr']*1000:.0f}ms")
    print(f"    Target per class  : {TARGET:,}")

    ic_rows, ii_rows = [], []
    ic_meta,  ii_meta  = [], []          # NEW: (subject_id, event_id) per row

    seizure_counter = {}   # NEW: running seizure index per subject, across all their files

    for rec in tqdm(records, desc="Processing EDFs"):
        subj_id = rec["filename"].split("_")[0]   # "chb01_03.edf" -> "chb01"
        file_start_sec = rec.get("file_start_sec", 0.0)  # NEW, see Patch 2

        data = rec["raw"].get_data()
        if data.shape[0] < n_ch:
            pad  = np.zeros((n_ch - data.shape[0], data.shape[1]))
            data = np.vstack([data, pad])
        data = data[:n_ch].astype(np.float32)
        T    = data.shape[1]
        total_recording_sec_accum = locals().get("total_recording_sec_accum", 0.0) + T / fs

        # ictal mask
        mask = np.zeros(T, dtype=np.uint8)
        for on, off in rec["seizure_times"]:
            s0 = max(0, int((on  - CFG["pre_ictal_ext"])  * fs))
            s1 = min(T, int((off + CFG["post_ictal_ext"]) * fs))
            mask[s0:s1] = 1

                # First, assign a persistent global seizure index to each seizure in this file
        # (fixes ev resetting to 0 per file — now unique per subject across all files)
        global_ev_for_local = []
        for on, off in rec["seizure_times"]:
            key = (subj_id, round(on, 3), round(off, 3))
            if key not in seizure_counter.get(subj_id, {}):
                seizure_counter.setdefault(subj_id, {})
                idx = len(seizure_counter[subj_id])
                seizure_counter[subj_id][key] = idx
            global_ev_for_local.append(seizure_counter[subj_id][key])

        # PASS 1: ictal windows (step = ic_str)
        for s in range(0, T - win, ic_str):
            if mask[s + win // 2] != 1:
                continue
            row = []
            for ch in data[:, s:s+win]:
                row.extend(extract_features(ch.astype(np.float64), fs))
            ic_rows.append(row)

            # which seizure interval does this window fall in? (local index within this file)
            local_ev = next((i for i,(on,off) in enumerate(rec["seizure_times"])
                       if int((on-CFG["pre_ictal_ext"])*fs) <= s+win//2
                       <= int((off+CFG["post_ictal_ext"])*fs)), 0)
            ev = global_ev_for_local[local_ev]   # NEW: globally unique per subject

            on, off = rec["seizure_times"][local_ev]
            # NEW: is this window inside the TRUE seizure (onset..offset), or only in the ±10s/5s margin?
            win_center_sec = (s + win // 2) / fs
            in_core_ictal = (on <= win_center_sec <= off)

            ic_meta.append((subj_id, f"{subj_id}_sz{ev}", s, s + win // 2, in_core_ictal))  # CHANGED

        # PASS 2: inter-ictal windows (step = ii_str)
        for s in range(0, T - win, ii_str):
            if mask[s + win // 2] != 0:
                continue
            row = []
            for ch in data[:, s:s+win]:
                row.extend(extract_features(ch.astype(np.float64), fs))
            ii_rows.append(row)
            ii_meta.append((subj_id, f"{subj_id}_bg", s, s + win // 2, False))  # CHANGED

        rec["raw"]._data = None

    n_ic = len(ic_rows)
    n_ii = len(ii_rows)
    print(f"\n  ── Real epochs collected ──────────────────────────")
    print(f"     Ictal       : {n_ic:>8,}")
    print(f"     Inter-ictal : {n_ii:>8,}")
    print(f"     Raw ratio   : 1 : {n_ii // max(n_ic, 1)}")
    # NEW: log true totals before undersampling, for FP/hour calculation later
    total_recording_sec = sum(r["raw"].get_data().shape[1] / fs for r in records)
    print(f"     Total interictal windows collected (pre-undersample): {n_ii:,}")
    print(f"     Total recording duration: {total_recording_sec/3600:.2f} hours")

    if n_ic == 0:
        raise RuntimeError(
            "No ictal epochs found — check data_dir and summary files.")

    rng    = np.random.default_rng(42)
    n_each = min(TARGET, n_ic, n_ii)

    print(f"\n  ── Balanced dataset ───────────────────────────────")
    if n_each < TARGET:
        print(f"     [INFO] Ictal cap: {n_ic:,}  "
              f"(target {TARGET:,}, short by {TARGET-n_each:,})")
        print(f"     [INFO] All rows are real EEG — no synthetic data added.")
    print(f"     Per class    : {n_each:,}")
    print(f"     Total rows   : {2*n_each:,}")
    print(f"     Feature cols : {n_ch * N_FEAT}  "
          f"({N_FEAT} per ch × {n_ch} ch)")
    print(f"       of which SNN: {8 * n_ch} cols  "
          f"(8 LIF features × {n_ch} ch)")

    ic_idx = rng.choice(n_ic, size=n_each, replace=False)
    ii_idx = rng.choice(n_ii, size=n_each, replace=False)

    X = np.concatenate([
        np.array([ic_rows[i] for i in ic_idx], dtype=np.float32),
        np.array([ii_rows[i] for i in ii_idx], dtype=np.float32),
    ], axis=0)
    y = np.concatenate([np.ones(n_each, dtype=np.int8), np.zeros(n_each, dtype=np.int8)])
    meta = [ic_meta[i] for i in ic_idx] + [ii_meta[i] for i in ii_idx]  # NEW

    perm = rng.permutation(len(y))
    X, y = X[perm], y[perm]
    meta = [meta[i] for i in perm]                                      # NEW
    # ── PATCH (R1-8): also return the FULL, un-undersampled dataset ──────
    # ic_rows/ii_rows/ic_meta/ii_meta already hold every extracted window,
    # before ic_idx/ii_idx randomly subsample down to n_each. Nothing above
    # this line is changed; we only keep what used to be discarded.
    X_full = np.concatenate([
        np.array(ic_rows, dtype=np.float32),
        np.array(ii_rows, dtype=np.float32),
    ], axis=0)
    y_full = np.concatenate([
        np.ones(len(ic_rows), dtype=np.int8),
        np.zeros(len(ii_rows), dtype=np.int8),
    ])
    meta_full = ic_meta + ii_meta
    # NOT shuffled: keeps window_start_sample meaningful as within-recording
    # order, which the natural-distribution / event-level analysis needs.

    return X, y, n_ch, meta, X_full, y_full, meta_full                  # CHANGED (3 new values)





In [ ]:
# ======================== SAVE ==================================
def save_csv(X, y, n_ch, meta):
    cols = [f"ch{c:02d}_{f}" for c in range(n_ch) for f in FEAT_NAMES]
    cols = cols[:X.shape[1]]

    df          = pd.DataFrame(X, columns=cols)
    df["label"] = y.astype(int)
    df["subject_id"]           = [m[0] for m in meta]
    df["event_id"]             = [m[1] for m in meta]
    df["window_start_sample"]  = [m[2] for m in meta]   # NEW
    df["window_center_sample"] = [m[3] for m in meta]   # NEW
    df["in_core_ictal"]        = [m[4] for m in meta]   # NEW — True only inside true onset..offset

    Path(CFG["output_csv"]).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(CFG["output_csv"], index=False)

    mb = Path(CFG["output_csv"]).stat().st_size / 1e6
    print(f"\n  Saved  →  {CFG['output_csv']}")
    print(f"  Shape  :  {df.shape[0]:,} rows × "
          f"{df.shape[1]} columns  ({mb:.1f} MB)")

    # Variance check — flag near-zero-variance columns
    feat_cols = [c for c in df.columns if c not in ("label","subject_id","event_id")]
    variances = df[feat_cols].var()

    
    low_var   = variances[variances < 1e-6]
    if len(low_var):
        print(f"\n  [WARN] {len(low_var)} near-zero-variance columns:")
        for col in low_var.index[:10]:
            print(f"    {col}  var={variances[col]:.2e}")
    else:
        print(f"  [OK]   All {len(variances)} feature columns have "
              f"sufficient variance.")

    # SNN feature summary
    snn_cols = [c for c in cols if "_snn_" in c]
    print(f"\n  SNN feature stats ({len(snn_cols)} columns):")
    print(df[snn_cols[:8]].describe().to_string())


# ── PATCH (R1-8): save the FULL, un-undersampled dataset ────────────────
# Same columns as save_csv(), so it drops straight into the existing
# preprocessing.py / load_raw() modelling pipeline with no code changes there.
# Does NOT modify save_csv() or its output — the balanced CSV is untouched.
def save_unbalanced_csv(X_full, y_full, n_ch, meta_full, output_path):
    cols = [f"ch{c:02d}_{f}" for c in range(n_ch) for f in FEAT_NAMES]
    cols = cols[:X_full.shape[1]]

    df          = pd.DataFrame(X_full, columns=cols)
    df["label"] = y_full.astype(int)
    df["subject_id"]           = [m[0] for m in meta_full]
    df["event_id"]             = [m[1] for m in meta_full]
    df["window_start_sample"]  = [m[2] for m in meta_full]
    df["window_center_sample"] = [m[3] for m in meta_full]
    df["in_core_ictal"]        = [m[4] for m in meta_full]

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)

    mb = Path(output_path).stat().st_size / 1e6
    print(f"\n  Saved UNBALANCED (natural class distribution)  →  {output_path}")
    print(f"  Shape  :  {df.shape[0]:,} rows × {df.shape[1]} columns  ({mb:.1f} MB)")
    print(f"  Class balance : {df['label'].mean():.4%} seizure "
          f"({int(df['label'].sum()):,} ictal / {int((df['label']==0).sum()):,} interictal)")

In [ ]:
# ======================== MAIN ==================================
def main():
    SEP = "=" * 65
    print(SEP)
    print("  CHB-MIT EEG — SNN-Enhanced Feature Extraction")
    print("  42 features per channel (34 original + 8 LIF/SNN)")
    print("  All rows are real EEG windows — no synthetic data")
    print(SEP)
    print(f"  Data   : {CFG['data_dir']}")
    print(f"  Output : {CFG['output_csv']}")
    print(f"  Feats  : {N_FEAT} per ch × {CFG['n_channels']} ch "
          f"= {N_FEAT * CFG['n_channels']} columns")
    print(f"  Target : up to {CFG['target_each']:,} + "
          f"{CFG['target_each']:,} = "
          f"{2*CFG['target_each']:,} rows")
    print(SEP)

    print("\n[1] Loading EDF files ...")
    records = CHBLoader(CFG["data_dir"]).load()

    print("\n[2] Extracting 42 features (incl. 8 SNN) per channel ...")
    X, y, n_ch, meta, X_full, y_full, meta_full = build_dataset(records)   # CHANGED (3 new values)

    print("\n[3] Saving CSV ...")
    save_csv(X, y, n_ch, meta)

    print("\n[3b] Saving UNBALANCED CSV (natural class distribution) ...")  # NEW
    save_unbalanced_csv(                                                    # NEW
        X_full, y_full, n_ch, meta_full,                                    # NEW
        output_path=str(Path(CFG["output_csv"]).parent /                   # NEW
                        "unbalanced_dataset_22ch_924feat_with_subjects.csv"),  # NEW
    )                                                                        # NEW

    print(f"\n{SEP}\n  Done.\n{SEP}")


if __name__ == "__main__":
    main()